# Notebook 04 — Insights & Visualisations

**Goal:** Produce all interactive Plotly charts and derive actionable personalization recommendations for each listener segment.

**Research question:** *What user segments exist based on listening diversity patterns, and how can streaming platforms use these insights to optimise personalization strategies for different listener types?*

**Inputs** (from `data/processed/`):
- `user_features.parquet`
- `cluster_labels.parquet`
- `scrobbles_updated.parquet`
- `artist_genres.parquet`
- `profiles.parquet`

**Outputs** (in `outputs/figures/`):
- `umap_clusters.html` — 2D UMAP scatter
- `cluster_heatmap.html` — feature profile heatmap
- `radar_chart.html` — radar comparison of segments
- `feature_importance.html` — discriminating features
- `genre_distribution.html` — genre breakdown per segment
- `temporal_heatmap_cluster_N.html` — listening patterns per segment

In [ ]:
import sys
import logging
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
logging.basicConfig(level=logging.INFO)

import pandas as pd
import numpy as np
import plotly.io as pio

pio.renderers.default = 'notebook'
Path('../outputs/figures').mkdir(parents=True, exist_ok=True)

In [ ]:
feature_matrix = pd.read_parquet('../data/processed/user_features.parquet')
labels_df      = pd.read_parquet('../data/processed/cluster_labels.parquet')
scrobbles      = pd.read_parquet('../data/processed/scrobbles_updated.parquet')
artist_genres  = pd.read_parquet('../data/processed/artist_genres.parquet')
profiles       = pd.read_parquet('../data/processed/profiles.parquet')

labels = labels_df['cluster'].values
cluster_names = dict(zip(labels_df['cluster'], labels_df['cluster_name']))
userids = labels_df['userid'].tolist()

print(f'Users: {len(userids)} | Segments: {labels_df["cluster"].nunique()}')
labels_df['cluster_name'].value_counts()

## 1. UMAP Cluster Scatter

In [ ]:
from src.visualization.plots import plot_umap_clusters, save_figure

umap_coords = labels_df[['umap_x', 'umap_y']].values

fig_umap = plot_umap_clusters(
    umap_coords=umap_coords,
    labels=labels,
    cluster_names=cluster_names,
    userids=userids,
)
save_figure(fig_umap, '../outputs/figures/umap_clusters')
fig_umap.show()

## 2. Cluster Feature Heatmap

Z-scored means — red = above average, blue = below average.

In [ ]:
from src.visualization.plots import plot_cluster_heatmap
from src.clustering.evaluation import summarise_clusters

# Focus on the most interpretable features for the heatmap
key_features = [
    'artist_entropy', 'genre_entropy', 'unique_artists', 'artist_concentration_20',
    'track_replay_rate', 'novelty_ratio', 'discovery_velocity_30d',
    'avg_tracks_per_session', 'avg_session_length_min', 'weekend_ratio',
    'temporal_hour_entropy', 'morning_ratio', 'evening_ratio',
    'mean_energy', 'mean_valence', 'mean_danceability', 'mean_acousticness',
]
key_features = [f for f in key_features if f in feature_matrix.columns]

fig_heatmap = plot_cluster_heatmap(
    feature_matrix=feature_matrix,
    labels=labels,
    features=key_features,
    cluster_names=cluster_names,
)
save_figure(fig_heatmap, '../outputs/figures/cluster_heatmap')
fig_heatmap.show()

## 3. Radar Chart — Segment Profiles

In [ ]:
from src.visualization.plots import plot_cluster_radar

cluster_summary = summarise_clusters(feature_matrix, labels)

radar_features = [
    'artist_entropy', 'genre_entropy', 'novelty_ratio',
    'track_replay_rate', 'avg_tracks_per_session', 'temporal_hour_entropy',
]
radar_features = [f for f in radar_features if f in feature_matrix.columns]

fig_radar = plot_cluster_radar(
    cluster_summary=cluster_summary,
    features=radar_features,
    cluster_names=cluster_names,
)
save_figure(fig_radar, '../outputs/figures/radar_chart')
fig_radar.show()

## 4. Feature Importance

In [ ]:
from src.clustering.evaluation import feature_importance
from src.visualization.plots import plot_feature_importance

imp_df = feature_importance(feature_matrix, labels)

fig_imp = plot_feature_importance(imp_df, top_n=20)
save_figure(fig_imp, '../outputs/figures/feature_importance')
fig_imp.show()

## 5. Genre Distribution by Segment

In [ ]:
from src.visualization.plots import plot_genre_distribution

fig_genre = plot_genre_distribution(
    scrobbles=scrobbles,
    artist_genres=artist_genres,
    labels=labels,
    userids=userids,
    top_n_genres=12,
    cluster_names=cluster_names,
)
save_figure(fig_genre, '../outputs/figures/genre_distribution')
fig_genre.show()

## 6. Temporal Listening Patterns (per segment)

In [ ]:
from src.visualization.plots import plot_temporal_heatmap

for cid in sorted(set(labels)):
    if cid == -1:
        continue
    fig_temp = plot_temporal_heatmap(
        scrobbles=scrobbles,
        labels=labels,
        userids=userids,
        cluster_id=cid,
        cluster_name=cluster_names.get(cid, f'Cluster {cid}'),
    )
    save_figure(fig_temp, f'../outputs/figures/temporal_heatmap_cluster_{cid}')
    fig_temp.show()

## 7. Business Insights & Personalization Recommendations

The cell below prints a structured summary of each segment's characteristics and actionable implications for streaming platforms.

In [ ]:
from src.clustering.evaluation import summarise_clusters, label_clusters

cluster_summary = summarise_clusters(feature_matrix, labels)
cluster_names_auto = label_clusters(cluster_summary, feature_matrix, labels)

# Key feature means per cluster (unscaled)
fm = feature_matrix.copy()
fm['cluster'] = labels

insight_features = [
    'artist_entropy', 'genre_entropy', 'unique_artists',
    'track_replay_rate', 'novelty_ratio', 'discovery_velocity_30d',
    'avg_tracks_per_session', 'weekend_ratio', 'temporal_hour_entropy',
]
insight_features = [f for f in insight_features if f in fm.columns]

global_means = fm[insight_features].mean()

print('=' * 72)
print('LISTENER SEGMENT PROFILES & PERSONALIZATION RECOMMENDATIONS')
print('=' * 72)

for cid in sorted(set(labels)):
    if cid == -1:
        continue
    cluster_users = fm[fm['cluster'] == cid]
    n = len(cluster_users)
    name = cluster_names.get(cid, f'Cluster {cid}')
    means = cluster_users[insight_features].mean()

    print(f'\nSEGMENT {cid}: {name.upper()}  ({n} users, {n/len(fm)*100:.1f}%)')
    print('-' * 60)

    for feat in insight_features:
        val = means[feat]
        gval = global_means[feat]
        delta = (val - gval) / (gval + 1e-9)
        arrow = '▲' if delta > 0.15 else ('▼' if delta < -0.15 else '—')
        print(f'  {feat:<35} {val:8.3f}  {arrow} (global: {gval:.3f})')

    # Heuristic recommendations
    recs = []
    if means.get('novelty_ratio', 0) > global_means.get('novelty_ratio', 0):
        recs.append('→ Prioritise new artist recommendations and discovery playlists')
    else:
        recs.append('→ Emphasise "More like your favourites" and artist radio')

    if means.get('track_replay_rate', 0) > global_means.get('track_replay_rate', 0) * 1.2:
        recs.append('→ Offer offline mode / download prompts for favourite tracks')

    if means.get('genre_entropy', 0) > global_means.get('genre_entropy', 0):
        recs.append('→ Cross-genre mood playlists and genre-blend features work well')
    else:
        recs.append('→ Deep-dive genre playlists and artist discography features')

    if means.get('weekend_ratio', 0) > 0.45:
        recs.append('→ Target weekend push notifications and curated weekend playlists')

    if means.get('avg_tracks_per_session', 0) > global_means.get('avg_tracks_per_session', 0) * 1.3:
        recs.append('→ Long-session features: auto-queuing, seamless transitions, sleep timer')

    print('\n  PLATFORM RECOMMENDATIONS:')
    for r in recs:
        print(f'  {r}')

print('\n' + '=' * 72)

## 8. Demographic Cross-Tabulation (Descriptive Only)

In [ ]:
demo = profiles.set_index('userid')[['gender', 'age', 'country']]
labeled = labels_df.set_index('userid').join(demo)

print('Gender distribution per segment:')
print(labeled.groupby('cluster_name')['gender'].value_counts(normalize=True).round(3))

print('\nMedian age per segment:')
print(labeled.groupby('cluster_name')['age'].median())

print('\nTop 3 countries per segment:')
print(
    labeled.groupby('cluster_name')['country']
    .value_counts()
    .groupby(level=0)
    .head(3)
)